# ログエラー検知ハンズオン (Microsoft Fabric)

Oracle アラートログ × Java アプリログから **エラーを検知する** ミニ基盤を、
Medallion アーキテクチャ (Bronze → Silver → Gold) で構築します。
Databricks 版と同じデータ・同じロジックを **Fabric の Lakehouse** に載せ替えたものです。

**ストーリー:** 2026-06-20 10:02 頃に Oracle 側でデッドロック (ORA-00060) → 接続枯渇
(ORA-12516 / ORA-12519) が発生し、ほぼ同時刻に Java アプリ側で `SQLException` /
コネクションプールのタイムアウトが急増するインシデントが起きています。
Gold 層で時間窓集計すると、この **DB 障害とアプリ障害の相関** が浮かび上がります。

| 層 | 役割 | このノートブックでの内容 |
|----|------|------------------------|
| **Bronze** | 生ログをそのまま保管 | ログファイルを1行=1レコードで Delta テーブル化 |
| **Silver** | 構造化・クレンジング | 正規表現でタイムスタンプ/レベル/エラーコードを抽出、スタックトレースを結合 |
| **Gold** | 集計・検知 | 時間窓ごとのエラー件数、しきい値超過アラート、DB×アプリ相関 |

> **前提:** Fabric 試用キャパシティ (または F SKU) が有効なワークスペースで実行してください。
> Lakehouse・ノートブックの実行には Trial / F キャパシティが必要です
> (Free ライセンス単体では動きません)。


## Step 0. Lakehouse を作成してノートブックにアタッチ

Databricks の「カタログ/スキーマ/ボリューム作成」に相当する準備を UI で行います。

1. ワークスペースで **新規 → Lakehouse** を選び、`loghandson` という名前で作成
2. このノートブックを開いた状態で、左パネルの **データソースの追加 / Lakehouse の追加** から
   `loghandson` を選び、**既定の Lakehouse としてアタッチ**
3. 左パネルの Lakehouse 内 **Files** を右クリック → **アップロード** で次の2ファイルを置く
   - `alert_ORCL.log` (Oracle アラートログ)
   - `app.log` (Java アプリログ)

既定 Lakehouse はノート実行ノードに `/lakehouse/default/` でマウントされ、
アップロードしたファイルは `/lakehouse/default/Files/raw/<ファイル名>` で読めます。
保存するテーブルは自動で既定 Lakehouse の `Tables` 配下に作られます
(`CREATE CATALOG` などの宣言は不要)。


In [ ]:
# パラメータ。Files 配下にアップロードしたフォルダ名に合わせる
RAW_PATH = "/lakehouse/default/Files/raw"

# アップロード済みファイルの確認
import os
print(os.listdir(RAW_PATH))

## Step 1. Bronze 層 — 生ログを1行=1レコードで取り込む

Bronze は「**加工せず、来たものをそのまま貯める**」層です。
ログの行順を後段のスタックトレース結合に使うため、`line_no`(行番号) を付けて取り込みます。


In [ ]:
def load_raw(filename: str):
    """Lakehouse Files 上のテキストログを (line_no, raw) の DataFrame として読み込む。"""
    path = f"{RAW_PATH}/{filename}"
    with open(path, "r", encoding="utf-8") as f:
        rows = [(i, line.rstrip("\n")) for i, line in enumerate(f)]
    return spark.createDataFrame(rows, "line_no INT, raw STRING")

bronze_oracle = load_raw("alert_ORCL.log")
bronze_app    = load_raw("app.log")

bronze_oracle.write.mode("overwrite").saveAsTable("bronze_oracle_raw")
bronze_app.write.mode("overwrite").saveAsTable("bronze_app_raw")

print("oracle rows:", bronze_oracle.count(), " app rows:", bronze_app.count())
display(bronze_app.orderBy("line_no").limit(8))

## Step 2. Silver 層① — Oracle アラートログの構造化

Oracle アラートログは **「タイムスタンプ行」+「本文行」のペア**で1イベントです。

```
2026-06-20T10:02:14.000000+09:00
ORA-00060: Deadlock detected. More info in file ...
```

行番号でグルーピングし、ヘッダ行とその後続行を1イベントに束ねてから、
タイムスタンプ・`ORA-xxxxx` コード・重大度を抽出します。


In [ ]:
from pyspark.sql import functions as F, Window

# ヘッダ行 = ISO タイムスタンプで始まる行
TS_HEADER = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}"

w = Window.orderBy("line_no")
df = (spark.table("bronze_oracle_raw")
        .withColumn("is_header", F.col("raw").rlike(TS_HEADER))
        # ヘッダ行の line_no を後続行に前方補完 → イベント ID
        .withColumn("event_id",
                    F.last(F.when(F.col("is_header"), F.col("line_no")), ignorenulls=True).over(w)))

# イベント単位に全行を結合
events = (df.groupBy("event_id")
            .agg(F.concat_ws("\n", F.collect_list("raw")).alias("full_text")))

silver_oracle = (events
    .withColumn("event_time",
        F.to_timestamp(F.regexp_extract("full_text",
            r"(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+)", 1),
            "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"))
    .withColumn("ora_code", F.regexp_extract("full_text", r"(ORA-\d{5})", 1))
    .withColumn("message",
        F.regexp_replace("full_text", r"^.*?\n", ""))   # 1行目(ts)を除いた本文
    .withColumn("severity",
        F.when(F.col("ora_code") != "", F.lit("ERROR")).otherwise(F.lit("INFO")))
    .select("event_time", "severity", "ora_code", "message")
    .filter(F.col("event_time").isNotNull()))

silver_oracle.write.mode("overwrite").saveAsTable("silver_oracle")
display(silver_oracle.filter("severity = 'ERROR'").orderBy("event_time"))

## Step 3. Silver 層② — Java アプリログの構造化 (マルチライン対応)

Java ログは1イベントが複数行になります (メッセージ行 + スタックトレース)。
ヘッダ行 (タイムスタンプ始まり) を起点にスタックトレースを結合し、
**スタックトレース内の `ORA-xxxxx` も抽出**します。これが後で DB ログとの相関キーになります。


In [ ]:
# ヘッダ行 = "yyyy-MM-dd HH:mm:ss,SSS LEVEL [thread] logger - msg"
APP_HEADER = r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3}\s"

w = Window.orderBy("line_no")
df = (spark.table("bronze_app_raw")
        .withColumn("is_header", F.col("raw").rlike(APP_HEADER))
        .withColumn("event_id",
                    F.last(F.when(F.col("is_header"), F.col("line_no")), ignorenulls=True).over(w)))

hdr = df.filter("is_header").select(F.col("event_id"), F.col("raw").alias("header"))
full = (df.groupBy("event_id")
          .agg(F.concat_ws("\n", F.collect_list("raw")).alias("full_text")))
joined = hdr.join(full, "event_id")

APP_RE = r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})\s+(\w+)\s+\[([^\]]+)\]\s+(\S+)\s+-\s+(.*)$"

silver_app = (joined
    .withColumn("event_time",
        F.to_timestamp(F.regexp_extract("header", APP_RE, 1), "yyyy-MM-dd HH:mm:ss,SSS"))
    .withColumn("level",  F.regexp_extract("header", APP_RE, 2))
    .withColumn("thread", F.regexp_extract("header", APP_RE, 3))
    .withColumn("logger", F.regexp_extract("header", APP_RE, 4))
    .withColumn("message", F.regexp_extract("header", APP_RE, 5))
    .withColumn("exception_class",
        F.regexp_extract("full_text", r"\n([a-zA-Z0-9_.]+(?:Exception|Error))", 1))
    .withColumn("ora_code", F.regexp_extract("full_text", r"(ORA-\d{5})", 1))
    .select("event_time", "level", "thread", "logger", "message",
            "exception_class", "ora_code")
    .filter(F.col("event_time").isNotNull()))

silver_app.write.mode("overwrite").saveAsTable("silver_app")
display(silver_app.filter("level = 'ERROR'").orderBy("event_time").limit(20))

## Step 4. Gold 層 — 時間窓ごとのエラー集計と急増検知

10分単位のウィンドウで ERROR 件数を集計します。平常時はほぼ 0〜1 件ですが、
インシデント時間帯だけ件数が跳ね上がるはずです。


In [ ]:
gold_app = (spark.table("silver_app")
    .filter("level = 'ERROR'")
    .groupBy(F.window("event_time", "10 minutes").alias("w"))
    .agg(F.count("*").alias("error_count"),
         F.collect_set("exception_class").alias("exception_types"),
         F.collect_set(F.when(F.col("ora_code") != "", F.col("ora_code"))).alias("related_ora"))
    .select(F.col("w.start").alias("window_start"),
            "error_count", "exception_types", "related_ora")
    .orderBy("window_start"))

gold_app.write.mode("overwrite").saveAsTable("gold_app_errors_10m")
display(gold_app.filter("error_count > 0"))

### しきい値ベースのアラート検知

「10分窓で ERROR が5件を超えたら異常」というシンプルなルールで検知します
(Fabric ノートブックの SQL は `%%sql` セルマジックを使います)。


In [ ]:
%%sql
SELECT
  window_start,
  error_count,
  exception_types,
  related_ora,
  CASE WHEN error_count > 5 THEN '🚨 ALERT' ELSE 'ok' END AS status
FROM gold_app_errors_10m
WHERE error_count > 5
ORDER BY window_start;

## Step 5. DB ログ × アプリログの相関分析

同じ10分窓で **Oracle 側エラー** と **アプリ側エラー** を横並びにし、`ORA-` コードで
突き合わせます。アプリの障害が DB のデッドロック / 接続枯渇に起因していることが
1枚のビューで見えます。


In [ ]:
%%sql
WITH ora AS (
  SELECT window(event_time, '10 minutes').start AS window_start,
         count(*) AS oracle_errors,
         collect_set(ora_code) AS oracle_ora_codes
  FROM silver_oracle WHERE severity = 'ERROR'
  GROUP BY 1
),
app AS (
  SELECT window(event_time, '10 minutes').start AS window_start,
         count(*) AS app_errors,
         collect_set(CASE WHEN ora_code <> '' THEN ora_code END) AS app_seen_ora_codes
  FROM silver_app WHERE level = 'ERROR'
  GROUP BY 1
)
SELECT
  coalesce(ora.window_start, app.window_start) AS window_start,
  coalesce(oracle_errors, 0) AS oracle_errors,
  coalesce(app_errors, 0)    AS app_errors,
  oracle_ora_codes,
  app_seen_ora_codes
FROM ora FULL OUTER JOIN app USING (window_start)
WHERE coalesce(oracle_errors,0) > 0 OR coalesce(app_errors,0) > 0
ORDER BY window_start;

**読み取り方:** インシデント窓 (10:00 台) では `oracle_errors` と `app_errors` が
同時に跳ね上がり、Oracle 側の `ORA-00060 / ORA-12516` が、アプリ側で観測された
`app_seen_ora_codes` と一致します → **DB のデッドロック＋接続枯渇がアプリ障害の根本原因**、
と切り分けられます。一方で 02:30 の `NullPointerException` (バッチ) は DB と無関係な単発障害、
と区別できます。


## Step 6. 発展課題 (Fabric ならではの拡張)

- **Power BI ダッシュボード:** Lakehouse の **SQL 分析エンドポイント** から
  `gold_app_errors_10m` に直接つなぎ、エラー件数の折れ線グラフを Power BI レポート化。
  Gold の Delta テーブルにそのままレポートを載せられるのが Fabric の強み。
- **Copilot / Q&A:** (試用版で Copilot が有効な場合) ノートブックや Power BI で
  「10時台に一番多かったエラーは?」と自然言語で問い合わせる。
- **Data pipeline / Dataflow Gen2:** Files にログが届いたら Bronze→Silver→Gold を
  自動実行するパイプラインを組み、増分処理にする。
- **検知ロジックの高度化:** 固定しきい値 (>5) を、過去N窓の移動平均 + 3σ に置き換える。
- **Data Activator (Reflex):** Gold テーブルのしきい値超過をトリガーに Teams / メール通知。

> 片付け: テーブルは Lakehouse の `Tables` から個別削除、または Lakehouse `loghandson`
> ごと削除すれば一括クリーンアップできます (試用キャパシティの節約)。
